# Run Cromite Google sign-in/Gemini arm64 build on Colab

This notebook prepares a temporary Colab runtime as a GitHub Actions self-hosted runner, then lets the queued `port-google-signin-gemini-arm64.yml` workflow build the APK. It uses the pushed feature branch and generates a temporary sample release keystore inside the workflow.

**Important:** Chromium Android builds are very large. Colab free runtimes commonly lack the required disk space, RAM, and session duration. Use a high-RAM/high-disk Colab runtime or a persistent VM. The notebook never stores a GitHub token in the file; use a Colab Secret named `GH_PAT`.

In [ ]:
# Configuration
REPO = 'vksp2k7/cromite'
BRANCH = 'port/google-signin-gemini-android'
CHROMIUM_TAG = '153.0.8010.37'
WORKSPACE = '/content/working_dir'
RUNNER_DIR = '/content/gh-runner'
RUNNER_LABELS = 'self-hosted,linux,x64,cromite-builder'
print(REPO, BRANCH, CHROMIUM_TAG)

In [ ]:
# Read the GitHub token from Colab Secrets, without printing it.
import os, getpass
try:
    from google.colab import userdata
    GH_PAT = userdata.get('GH_PAT')
except Exception:
    GH_PAT = None
if not GH_PAT:
    GH_PAT = getpass.getpass('GitHub PAT with repository Actions runner registration permission: ')
if not GH_PAT:
    raise RuntimeError('A GitHub PAT is required. Store it as the Colab Secret GH_PAT.')
print('GitHub token loaded without displaying its value.')

In [ ]:
# Install build prerequisites and inspect the Colab runtime.
!sudo apt-get update -qq
!sudo DEBIAN_FRONTEND=noninteractive apt-get install -y -qq git curl unzip python3 openjdk-21-jdk-headless build-essential clang ninja-build
!df -h /content
!free -h
import shutil
free_gb = shutil.disk_usage('/content').free / 1024**3
if free_gb < 120:
    raise RuntimeError(f'Only {free_gb:.1f} GB free. A Chromium build needs at least 120 GB; use a larger runtime or persistent VM.')
print(f'{free_gb:.1f} GB free; continuing.')

In [ ]:
# Fetch depot_tools and the exact Chromium release source expected by Cromite.
import os, subprocess, textwrap
os.makedirs(WORKSPACE, exist_ok=True)
subprocess.run(['bash','-lc', f'rm -rf {WORKSPACE}/depot_tools && git clone --depth=1 https://chromium.googlesource.com/chromium/tools/depot_tools.git {WORKSPACE}/depot_tools'], check=True)
os.environ['PATH'] = f"{WORKSPACE}/depot_tools:" + os.environ['PATH']
os.environ['DEPOT_TOOLS_UPDATE'] = '0'
src = f'{WORKSPACE}/chromium/src'
if not os.path.exists(f'{src}/.git'):
    subprocess.run(['bash','-lc', f'rm -rf {WORKSPACE}/chromium && mkdir -p {src} && cd {src} && git init && git remote add origin https://chromium.googlesource.com/chromium/src.git && git fetch --depth=2 origin refs/tags/{CHROMIUM_TAG}:refs/tags/{CHROMIUM_TAG} && git checkout {CHROMIUM_TAG}'], check=True)
sha = subprocess.check_output(['git','-C',src,'rev-parse',CHROMIUM_TAG], text=True).strip()
gclient = "solutions = [\n  {\"name\": \"src\", \"url\": \"https://chromium.googlesource.com/chromium/src.git@%s\", \"deps_file\": \"DEPS\", \"managed\": False}\n]\ntarget_os = [\"android\"]\n" % sha
open(f'{WORKSPACE}/.gclient','w').write(gclient)
subprocess.run(['bash','-lc', f'cd {WORKSPACE} && gclient sync -D --no-history --nohooks'], check=True)
subprocess.run(['bash','-lc', f'cd {src} && gclient runhooks'], check=True)
print('Chromium source and hooks prepared at', src)

In [ ]:
# Make the workflow's fixed /home/lg/working_dir path point to this Colab workspace.
!sudo mkdir -p /home/lg
!sudo ln -sfn /content/working_dir /home/lg/working_dir
!test -d /home/lg/working_dir/chromium/src/.git && echo 'Chromium source path is ready.'

In [ ]:
# Register this Colab runtime as the temporary cromite-builder runner.
import requests, json, os, platform, time, tarfile, io, pathlib
headers = {'Accept':'application/vnd.github+json', 'Authorization':f'Bearer {GH_PAT}', 'X-GitHub-Api-Version':'2022-11-28'}
registration = requests.post(f'https://api.github.com/repos/{REPO}/actions/runners/registration-token', headers=headers, timeout=30)
registration.raise_for_status()
runner_token = registration.json()['token']
release = requests.get('https://api.github.com/repos/actions/runner/releases/latest', headers={'Accept':'application/vnd.github+json'}, timeout=30)
release.raise_for_status()
asset = next(a for a in release.json()['assets'] if a['name'].startswith('actions-runner-linux-x64-') and a['name'].endswith('.tar.gz'))
os.makedirs(RUNNER_DIR, exist_ok=True)
archive = requests.get(asset['browser_download_url'], timeout=120)
archive.raise_for_status()
open('/content/runner.tar.gz','wb').write(archive.content)
with tarfile.open('/content/runner.tar.gz') as tf: tf.extractall(RUNNER_DIR)
runner_name = 'colab-cromite-' + str(int(time.time()))
subprocess.run(['bash','-lc', f"cd {RUNNER_DIR} && ./config.sh --unattended --url https://github.com/{REPO} --token {runner_token} --name {runner_name} --labels {RUNNER_LABELS} --work _work --replace"], check=True)
print('Runner registered:', runner_name)
print('The next cell must remain running while GitHub Actions builds the APK.')

In [ ]:
# Start the runner. The already queued workflow should be picked up automatically.
# Keep this cell running until the GitHub Actions job completes and uploads its artifact.
import subprocess
subprocess.run(['bash','-lc', f'cd {RUNNER_DIR} && ./run.sh'], check=False)

## After completion

Open the GitHub Actions run for `Port Google sign-in and Gemini — Android arm64` and download the artifact named `cromite-google-signin-gemini-arm64-sample-release`. The APK is signed with an ephemeral sample key generated by the workflow and is not suitable for production updates.

If the Colab runtime disconnects, the self-hosted runner stops and the GitHub job will remain queued or fail. Re-run the setup cells on a fresh runtime.